# 01: データ前処理

CSVデータの読み込みとクリーニング、基本統計情報の確認を行います。

## 目的
- 過去当選番号データの読み込み
- データの基本統計情報の確認
- 学習用データセットの準備（設定ファイルで指定された回数分）

## 基準回号
- **基準回号**: 最新回号を自動取得（`config.py`で設定可能）
- この回号を基準にさかのぼって学習データを構築します

## 設定ファイル
- `config.py`で学習範囲（TRAIN_SIZE）を設定できます
- デフォルト: 1000回分


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# 設定ファイルをインポート
import sys
sys.path.append(str(Path(__file__).parent if '__file__' in globals() else Path.cwd()))
from config import TRAIN_SIZE, BASE_ROUND_AUTO, BASE_ROUND, TRAIN_DATA_CSV

# プロジェクトルートのパスを設定
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"データディレクトリ: {DATA_DIR}")
print(f"\n学習設定:")
print(f"  学習範囲: {TRAIN_SIZE}回分")
print(f"  基準回号自動取得: {BASE_ROUND_AUTO}")
if not BASE_ROUND_AUTO:
    print(f"  基準回号（手動指定）: {BASE_ROUND}")


In [ ]:
# CSVデータの読み込み
csv_path = DATA_DIR / 'past_results.csv'

if not csv_path.exists():
    raise FileNotFoundError(f"データファイルが見つかりません: {csv_path}")

df = pd.read_csv(csv_path)

# 当選番号を文字列型に変換（数値型で読み込まれる可能性があるため）
df['n3_winning'] = df['n3_winning'].astype(str).str.replace('.0', '', regex=False)
df['n4_winning'] = df['n4_winning'].astype(str).str.replace('.0', '', regex=False)

# 先頭0を補完（数値型として読み込まれた場合、先頭0が失われるため）
df['n3_winning'] = df['n3_winning'].apply(lambda x: str(x).zfill(3) if pd.notna(x) and str(x) != 'NULL' else x)
df['n4_winning'] = df['n4_winning'].apply(lambda x: str(x).zfill(4) if pd.notna(x) and str(x) != 'NULL' else x)

# リハーサル数字を文字列型に変換（数値型で読み込まれる可能性があるため）
# NULL値の場合はそのまま維持（NaNは文字列変換時にも保持される）
if 'n3_rehearsal' in df.columns:
    df['n3_rehearsal'] = df['n3_rehearsal'].astype(str).str.replace('.0', '', regex=False)
    # NULL値やNaNの場合はそのまま、有効な値の場合のみ先頭0を補完
    df['n3_rehearsal'] = df['n3_rehearsal'].apply(
        lambda x: str(x).zfill(3) if pd.notna(x) and str(x) != 'NULL' and str(x) != 'nan' and str(x) != '' else x
    )
    # 'nan'文字列をNaNに戻す（NULL値として扱う）
    df['n3_rehearsal'] = df['n3_rehearsal'].replace('nan', pd.NA).replace('NULL', pd.NA).replace('', pd.NA)

if 'n4_rehearsal' in df.columns:
    df['n4_rehearsal'] = df['n4_rehearsal'].astype(str).str.replace('.0', '', regex=False)
    # NULL値やNaNの場合はそのまま、有効な値の場合のみ先頭0を補完
    df['n4_rehearsal'] = df['n4_rehearsal'].apply(
        lambda x: str(x).zfill(4) if pd.notna(x) and str(x) != 'NULL' and str(x) != 'nan' and str(x) != '' else x
    )
    # 'nan'文字列をNaNに戻す（NULL値として扱う）
    df['n4_rehearsal'] = df['n4_rehearsal'].replace('nan', pd.NA).replace('NULL', pd.NA).replace('', pd.NA)

print(f"データ行数: {len(df)}")
print(f"\nデータの最初の5行:")
print(df.head())
print(f"\nデータの基本情報:")
print(df.info())
print(f"\nリハーサル数字の有無:")
if 'n3_rehearsal' in df.columns:
    print(f"  N3リハーサル: {df['n3_rehearsal'].notna().sum()}件 / {len(df)}件")
if 'n4_rehearsal' in df.columns:
    print(f"  N4リハーサル: {df['n4_rehearsal'].notna().sum()}件 / {len(df)}件")


In [ ]:
# データクリーニング
# 1. 必須フィールドの確認
required_columns = ['round_number', 'draw_date', 'n3_winning', 'n4_winning']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"必須カラムが見つかりません: {missing_columns}")

# 2. NULL値の処理
df = df.dropna(subset=required_columns)

# 3. 回号の検証
df = df[df['round_number'].between(1, 9999)]

# 4. 当選番号のフォーマット検証
df = df[df['n3_winning'].str.match(r'^\d{3}$', na=False)]
df = df[df['n4_winning'].str.match(r'^\d{4}$', na=False)]

# 5. 回号の降順でソート（最新が先頭）
df = df.sort_values('round_number', ascending=False).reset_index(drop=True)

print(f"クリーニング後のデータ行数: {len(df)}")
print(f"\n回号の範囲: {df['round_number'].min()} 〜 {df['round_number'].max()}")


In [ ]:
# 基準回号の確認（設定ファイルから読み込み）
if BASE_ROUND_AUTO:
    BASE_ROUND = df['round_number'].max()  # 最新回号を自動取得
else:
    if BASE_ROUND is None:
        raise ValueError("BASE_ROUND_AUTOがFalseの場合、BASE_ROUNDを設定してください")

base_row = df[df['round_number'] == BASE_ROUND]

if len(base_row) == 0:
    raise ValueError(f"基準回号 {BASE_ROUND} が見つかりません")

BASE_DATE = base_row.iloc[0]['draw_date']

print(f"基準回号: {BASE_ROUND}")
print(f"基準日: {BASE_DATE}")
print(f"\n基準回号のデータ:")
print(base_row[['round_number', 'draw_date', 'n3_winning', 'n4_winning']])


In [ ]:
# 学習用データセットの準備（設定ファイルから学習範囲を読み込み）
# 基準回号（最新回号）からさかのぼって指定回数分を取得

# 基準回号のインデックスを取得
base_idx = df[df['round_number'] == BASE_ROUND].index[0]

# 基準回号を含む指定回数分を取得
train_df = df.iloc[base_idx:base_idx+TRAIN_SIZE].copy()

# 回号の昇順でソート（古い順）
train_df = train_df.sort_values('round_number', ascending=True).reset_index(drop=True)

print(f"学習用データセット: {len(train_df)}回分")
print(f"回号範囲: {train_df['round_number'].min()} 〜 {train_df['round_number'].max()}")
print(f"\n最初の5行:")
print(train_df[['round_number', 'draw_date', 'n3_winning', 'n4_winning']].head())
print(f"\n最後の5行:")
print(train_df[['round_number', 'draw_date', 'n3_winning', 'n4_winning']].tail())


In [ ]:
# データの基本統計情報
print("=== N3当選番号の統計 ===")
n3_digits = train_df['n3_winning'].str.split('', expand=True).iloc[:, 1:4].astype(int)
print(f"各桁の平均値:")
print(n3_digits.mean())
print(f"\n各桁の出現頻度（0-9）:")
for i in range(3):
    col_idx = i + 1  # split結果の列インデックス（1, 2, 3）
    print(f"  桁{i+1}:")
    print(n3_digits.iloc[:, i].value_counts().sort_index())

print("\n=== N4当選番号の統計 ===")
n4_digits = train_df['n4_winning'].str.split('', expand=True).iloc[:, 1:5].astype(int)
print(f"各桁の平均値:")
print(n4_digits.mean())
print(f"\n各桁の出現頻度（0-9）:")
for i in range(4):
    col_idx = i + 1  # split結果の列インデックス（1, 2, 3, 4）
    print(f"  桁{i+1}:")
    print(n4_digits.iloc[:, i].value_counts().sort_index())


In [ ]:
# 罫線マスターデータの読み込み確認
keisen_path = DATA_DIR / 'keisen_master.json'

if not keisen_path.exists():
    raise FileNotFoundError(f"罫線マスターファイルが見つかりません: {keisen_path}")

with open(keisen_path, 'r', encoding='utf-8') as f:
    keisen_master = json.load(f)

print("罫線マスターデータの構造:")
print(f"  N3: {list(keisen_master.get('n3', {}).keys())}")
print(f"  N4: {list(keisen_master.get('n4', {}).keys())}")

# サンプルデータの確認
if 'n3' in keisen_master and '百の位' in keisen_master['n3']:
    sample_key = list(keisen_master['n3']['百の位'].keys())[0]
    sample_value = keisen_master['n3']['百の位'][sample_key]
    print(f"\nサンプル（N3百の位、前々回={sample_key}）:")
    print(f"  キー数: {len(sample_value)}")
    if sample_value:
        first_key = list(sample_value.keys())[0]
        print(f"  前回={first_key}の場合の予測出目: {sample_value[first_key]}")


In [ ]:
# データセットの保存（次のNotebookで使用）
# 学習用データセットをCSV形式で保存（設定ファイルからファイル名を取得）

# CSV保存前に、リハーサル数字のNaN値を空文字列に置き換え、小数点を削除
train_df_save = train_df.copy()
if 'n3_rehearsal' in train_df_save.columns:
    # NaN値を空文字列に置き換え
    train_df_save['n3_rehearsal'] = train_df_save['n3_rehearsal'].fillna('')
    # 文字列型に変換して小数点を削除
    train_df_save['n3_rehearsal'] = train_df_save['n3_rehearsal'].astype(str).str.replace('.0', '', regex=False)
    # 'nan'文字列を空文字列に置き換え
    train_df_save['n3_rehearsal'] = train_df_save['n3_rehearsal'].replace('nan', '')

if 'n4_rehearsal' in train_df_save.columns:
    # NaN値を空文字列に置き換え
    train_df_save['n4_rehearsal'] = train_df_save['n4_rehearsal'].fillna('')
    # 文字列型に変換して小数点を削除
    train_df_save['n4_rehearsal'] = train_df_save['n4_rehearsal'].astype(str).str.replace('.0', '', regex=False)
    # 'nan'文字列を空文字列に置き換え
    train_df_save['n4_rehearsal'] = train_df_save['n4_rehearsal'].replace('nan', '')

# CSV保存（NaN値は空文字列として保存）
train_csv_path = DATA_DIR / TRAIN_DATA_CSV
train_df_save.to_csv(train_csv_path, index=False, na_rep='')
print(f"学習用データセットを保存しました: {train_csv_path}")
print(f"ファイルサイズ: {train_csv_path.stat().st_size / 1024:.2f} KB")

# 保存されたデータの確認（小数点が付いていないか確認）
saved_df = pd.read_csv(train_csv_path)
if 'n3_rehearsal' in saved_df.columns:
    sample_rows = saved_df[saved_df['n3_rehearsal'].notna() & (saved_df['n3_rehearsal'] != '')]
    if len(sample_rows) > 0:
        sample_rehearsal = sample_rows['n3_rehearsal'].iloc[0]
        print(f"\n保存後のN3リハーサル数字サンプル: {sample_rehearsal}")
        print(f"  型: {type(sample_rehearsal)}")
        if '.' in str(sample_rehearsal):
            print(f"  ⚠️ 警告: 小数点が含まれています！")


## 次のステップ

データ前処理が完了しました。次のNotebook (`02_chart_generation.ipynb`) で、各回号に対して4パターン（A1/A2/B1/B2）の予測表を生成します。
